# Parameter-efficient fine-tuning with LoRA
Low-rank adaptation (LoRA) is a technique that adapts a pretrained model to bettter suit a specific, often smaller dataset by adjusting only a small subset of the model's weight parameters.

## Finetuning GTP2(355M) model to follow instruction using LoRA

In [3]:
# Initializing pre-trained model

from gpt_download import download_and_load_gpt2
from utils.gpt_model import GPTModel
from utils.weight_loader import load_weights_into_gpt

model_size = "355M"
model_configs = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
    "emb_dim": 1024,
    "n_layers": 24,
    "n_heads": 16
}

settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")
model = GPTModel(model_configs)
load_weights_into_gpt(model, params)
model.eval();

2025-09-20 17:01:59.539717: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


File already exists and is up-to-date: gpt2/355M/checkpoint
File already exists and is up-to-date: gpt2/355M/encoder.json
File already exists and is up-to-date: gpt2/355M/hparams.json
File already exists and is up-to-date: gpt2/355M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/355M/model.ckpt.index
File already exists and is up-to-date: gpt2/355M/model.ckpt.meta
File already exists and is up-to-date: gpt2/355M/vocab.bpe


### Preparing dataset
We will be using the Alpaca dataset, by researchers at Standford, which is one of the earliest and most popular openly shared instruction datasets

In [4]:
# Downloading the dataset
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode('utf-8')
        with open(file_path, 'w', encoding='utf-8') as file:
            file.write(text_data)
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

file_path = 'alpaca-instruct-data.json'
url = (
    "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 52002


In [5]:
# Inspect some sample entries in the dataset
print(json.dumps(data[50], indent=2))
print(json.dumps(data[1001], indent=2))

{
  "instruction": "Edit the following sentence to make it more concise.",
  "input": "He ran to the bus stop in order to catch the bus that was due to arrive in five minutes.",
  "output": "He ran to the bus stop, due to arrive in five minutes."
}
{
  "instruction": "Compare and contrast the effects of individual and societal performance",
  "input": "",
  "output": "Individual performance is related to the actions and decisions made by individuals while societal performance focuses on the actions and decisions of an entire group or organization. Individual performance is more focused on the needs and goals of the individual while societal performance is based on collective goals. Individual performance can be more easily measured and quantified while measuring the impact of societal performance can be more difficult as it usually involves evaluating the individual components making up the larger picture."
}


In [6]:
# Implementing prompt formatting function to convert dataset entries into Alpaca-style input formats

def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""
    )

    return instruction_text + input_text

In [7]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Edit the following sentence to make it more concise.

### Input:
He ran to the bus stop in order to catch the bus that was due to arrive in five minutes.

### Response:
He ran to the bus stop, due to arrive in five minutes.


In [8]:
# Splitting the dataset into training, validation, and test sets

train_portion = int(len(data) * 0.85) # 85% for training
test_portion = int(len(data) * 0.10)   # 10% for testing
val_portion = len(data) - train_portion - test_portion # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 44201
Validation set length: 2601
Test set length: 5200


### Organizing data into training batches

In [9]:
# InstructionDataset class which applies formating the input and pretokenize

import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )
    
    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [10]:
# We will use the same tokenizer as for GPT-2
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

Batching process for instruction fine-tuning is bit more complex than batching process of pretraining on unlabled text corpus. In pre-training PyTorch Dataloader class can be used as it is which has a default `collate` function to combine list of samples into batches.
Due to the structural change in instruction-data set, we will create a custom `collate` function for efficient batching

In [11]:
# Implementing a comprehensive custom batch collate function with following functionalities:
# 1. Padding sequences in the batch to the same length while allowing different batches to have different lengths.
# 2. Generating target token ids by shifting the input token ids one position to the right and appending an end-of-text token.
# 3. Replacing all padding tokens except the first one in the target sequences with a specified ignore_index value (-100) to ensure that the model does not learn from padding tokens.
#    PyTorch's CrossEntropyLoss function ignores target values set to ignore_index, -100 during loss computation.
# 4. Optionally truncating sequences to a specified maximum length.

# Notice that, we are adding an extra padding token at the end of each sequence before padding to ensure that there is at least one padding token in the target sequence. This is important for sequences that are exactly the maximum length.

def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] * (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # Removes extra padding token added earlier
        targets = torch.tensor(padded[1:])  # Shifted version of inputs

        # Replace all padding tokens except the first one in targets by ignore_index. This is to ensure that the model does not learn from padding tokens.
        mask = (targets == pad_token_id)
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None: # Optionally truncates to the maximum sequence length
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [12]:
# Test custom_collate_fn
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = [inputs_1, inputs_2, inputs_3]
input_tensor, target_tensor = custom_collate_fn(batch)
print("Input Tensor:\n", input_tensor)
print("Target Tensor:\n", target_tensor)

Input Tensor:
 tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
Target Tensor:
 tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [13]:
#### Creating DataLoader instances for training, validation, and test sets.

from functools import partial
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Creating a partial function for the custom collate function with device and allowed_max_length parameter
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

num_workers = 0
batch_size  = 8
torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers,
    collate_fn=customized_collate_fn
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
    collate_fn=customized_collate_fn
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
    collate_fn=customized_collate_fn
)

### Implementing a LoRA Layer
We will create a LoRA Layer that creates the matrices A and B, along with the `alpha` scaling factor and the `rank (r)`
The `rank` governs the number of extra parameters introduced by LoRA, creates balance between the adaptability of the model and its efficiency via the number of parameters used.
The `alpha` functions as a scaling factor for the output from the low-rank adaptation. It primarily dictates the degree to which the output from the adapter layer can affect the original layer's output.

In [14]:
import math

class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)) # Kaiming uniform initialization used for Linear layers in PyTorch
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        x = self.alpha * (x @ self.A @ self.B)
        return x

In [15]:
# Creating a Linear layer with LoRA adaptation
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            in_dim=linear.in_features,
            out_dim=linear.out_features,
            rank=rank,
            alpha=alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [16]:
# Creating a function to replace Linear layers in the model with LinearWithLoRA layers
def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            new_module = LinearWithLoRA(module, rank, alpha)
            setattr(model, name, new_module)
        else:
            replace_linear_with_lora(module, rank, alpha)
    return model

In [17]:
# Before applying LinearWithLoRA layers, first freeze all model parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters before LoRA: {total_params:,}")
for param in model.parameters():
    param.requires_grad = False
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters after freezing: {total_params:,}")


Total trainable parameters before LoRA: 406,286,336
Total trainable parameters after freezing: 0


In [18]:
# Replace Linear layers with LinearWithLoRA layers
rank = 16
alpha = 16
model = replace_linear_with_lora(model, rank, alpha)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters after applying LoRA: {total_params:,}")

Total trainable parameters after applying LoRA: 7,898,384


In [19]:
from utils.gpt_training import train_model_simple, calc_loss_loader

# Calculating initial loss for training and validation sets
model.to(device)
torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("Training Loss:", train_loss)
print("Validation Loss:", val_loss)

Training Loss: 3.267276668548584
Validation Loss: 3.460875701904297


In [ ]:
# Start finetuning the model with LoRA layers

import time
start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
num_epochs = 10

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes")

In [ ]:
# Plotting losses
from utils.plots_helper import plot_losses

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

In [ ]:
# Save LoRA weights separately
lora_state_dict = {}
for name, param in model.named_parameters():
    if 'lora' in name:
        lora_state_dict[name] = param.cpu().clone()
torch.save(lora_state_dict, "lora_weights_instruct_gtp2_355.pth")